# 挑战：用本地量化模型生成 REST 示例 JSON

## 练习目标（理念）

构建一个**综合数据集生成器**：根据 OpenAPI schema 生成开发 REST API 时可用的示例 JSON。

- 用多种模型与提示尝试不同输出
- 尝试 4-bit 量化（BitsAndBytes）以节省显存
- 本笔记本聚焦「从 OpenAPI 抽 schema → 多模型生成 → JSON/Schema 校验」

## 和本课的关系

| 概念 | 本练习里你会看到 |
|------|------------------|
| Hugging Face Transformers | `AutoModelForCausalLM` / `AutoTokenizer` |
| 量化推理 | `BitsAndBytesConfig`（4-bit NF4） |
| 结构化输出 | 按 JSON Schema 生成并校验 |
| OpenAPI | 从 `storefront-sample.json` 提取 requestBody schema |

## 怎么跑

1. 需要 **NVIDIA GPU + CUDA**（不支持纯 CPU）
2. 准备 `.env`：`HF_TOKEN`（Llama 需先在 Hugging Face 申请访问）
3. 确保同目录有 `storefront-sample.json`
4. 从上到下依次运行单元格


## ⚠️ 运行前必读

**需要 GPU。** 本笔记本通过 BitsAndBytes 做 4-bit 量化推理，需要支持 CUDA 的 NVIDIA GPU；不支持 CPU 推理。

**磁盘空间。** 模型首次运行时从 Hugging Face 下载并缓存到 `~/.cache/huggingface/hub`（每个模型一次性）：

| 模型 | 下载大小（大约） |
|------|------------------|
| Llama 3.1 8B Instruct | ~16 GB |
| Qwen 2.5 Coder 7B Instruct | ~15 GB |
| Phi 4 Mini Instruct | ~5 GB |

**合计：首次约 36 GB**；之后从本地缓存加载。

**HF Token。** Llama 需要 Hugging Face 账号，并在 [meta-llama/Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct) 获得访问批准。在 `.env` 中设置 `HF_TOKEN`。

---

## 安装顺序很重要

先装带 CUDA 的 PyTorch，再用 `--no-deps` 装其余包，避免它们把 CPU 版 torch 拉进来：

```bash
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
uv pip install --no-deps accelerate bitsandbytes transformers==4.57.6 huggingface_hub[hf_xet]
```

本地环境里这些安装是持久的——通常只需跑一次。


In [ ]:
# ========== 版本自检：确认 CUDA 相关依赖已装好 ==========

# 导入 torch / accelerate / bitsandbytes / transformers：量化推理与模型加载栈
import torch, accelerate, bitsandbytes, transformers

# 打印各库版本，便于排查「装错 CPU 版 torch」等问题
print(f"torch=={torch.__version__}")
print(f"accelerate=={accelerate.__version__}")
print(f"bitsandbytes=={bitsandbytes.__version__}")
print(f"transformers=={transformers.__version__}")


In [ ]:
# ========== 导入：模型、环境变量、Hugging Face 登录 ==========

# 从 transformers 导入：pipeline（备用）、因果 LM、分词器、4-bit 量化配置
from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
# 标准库 os：读环境变量（Environment Variables），例如 HF_TOKEN
import os
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# OpenAI 客户端：本笔记本主路径用本地 HF 模型；保留导入以兼容扩展
from openai import OpenAI
# huggingface_hub.login：用 token 登录，才能拉受控模型（如 Llama）
from huggingface_hub import login

# 加载 .env；override=True 表示用文件覆盖已有同名环境变量
load_dotenv(override=True)


In [ ]:
# ========== 常量：模型 ID 与 OpenAPI 文件路径集中管理 ==========

# Llama 3.1 8B Instruct：需 HF 授权；字符串必须与 Hub 上的 repo id 一致
LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# Qwen2.5 Coder：偏代码/结构化生成，适合 JSON 任务
QWEN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
# Phi-4 Mini：更小、下载更快，用作对照
PHI_MODEL = "microsoft/Phi-4-mini-instruct"
# 本地 OpenAPI 样例文件：含 paths / requestBody / schema
OPENAPI_FILE = "storefront-sample.json"


In [ ]:
# ========== Hugging Face 登录：拉取受控模型前先鉴权 ==========

# 用环境变量 HF_TOKEN 登录；add_to_git_credential=True 写入本机凭证缓存
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)


In [ ]:
# ========== 解析 OpenAPI $ref：把引用解析成真实 schema 对象 ==========


# 按 JSON Pointer 风格路径（去掉 #/ 后按 / 分段）在 spec 里逐层取值
def resolve_ref(spec, ref):
    # 把 "#/components/schemas/Foo" 变成 ["components", "schemas", "Foo"]
    path = ref.replace("#/", "").split("/")
    # 从整份 OpenAPI 文档根开始
    obj = spec
    # 沿路径下钻，直到拿到被引用的对象
    for p in path:
        obj = obj[p]
    return obj


In [ ]:
# ========== 从 OpenAPI 提取某个端点的 JSON Schema ==========

# 标准库 json：读 OpenAPI 文件、美化打印 schema
import json


# 按 path + HTTP method 取出 requestBody 里 application/json 的 schema
def extract_schema(openapi, path, method):
    # OpenAPI paths 里 method 键通常是小写（get/post/...）
    method = method.lower()
    # 定位到具体 operation 对象
    endpoint = openapi["paths"][path][method]
    # 取出 JSON 请求体对应的 schema（可能仍是 $ref）
    schema = endpoint["requestBody"]["content"]["application/json"]["schema"]
    return schema


# 注意：这里有意只抽一个端点，保持笔记本最小可复现
# extract_schema 本身可用于任意 path/method；单端点只是演示选择
ENDPOINT_PATH = "/cart/items"
ENDPOINT_METHOD = "post"

# 打开本地 OpenAPI 样例并解析为 dict
with open(OPENAPI_FILE) as f:
    spec = json.load(f)

# 提取该端点的 request schema
schema = extract_schema(spec, ENDPOINT_PATH, ENDPOINT_METHOD)

# 若 schema 是 $ref，则解析成完整对象，方便后续塞进 prompt / 校验
if "$ref" in schema:
    schema = resolve_ref(spec, schema["$ref"])

# 打印美化后的 schema，便于肉眼确认字段
print(json.dumps(schema, indent=2))


In [ ]:
# ========== 构造 chat messages：system 定角色，user 带 schema ==========

# 有意保持 prompt 极简：只要求「按 schema 生成真实感 JSON」
# 注意：prompt 字符串保持英文原样（影响模型行为，禁止翻译）

user_prompt = f"""Generate realistic JSON objects following this schema:


{schema}
"""


system_prompt = """


You excel at generating realistic JSON objects following a given schema.
"""


# Chat Completions 风格消息列表：后面 apply_chat_template 会用到
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]


In [ ]:
# ========== 4-bit 量化加载 + 生成 JSON 文本 ==========

# 再次导入 torch：本格配置 bfloat16 计算 dtype 时需要
import torch

# BitsAndBytes 4-bit 配置：省显存；NF4 + double quant 是常见组合
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


# 按模型名加载分词器与因果语言模型（自动设备映射 + 量化）
def load_model(model_name):
    # 从 Hub 拉 tokenizer（含 chat_template，若模型提供）
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 部分模型没有 pad_token：用 eos 顶上，避免 generate 报错
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # device_map="auto" 让 accelerate 自动把层放到可用 GPU
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", quantization_config=quant_config
    )
    return model, tokenizer


# 用 chat template 编码 messages，在 CUDA 上采样生成，只解码新 token
def generate_json(model, tokenizer, messages, max_new_tokens=1000):
    # 把 system/user 编成模型偏好的对话格式，并移到 GPU
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to("cuda")

    # 调试信息：输入长度、是否找到 chat_template
    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    print(f"  Chat template found: {tokenizer.chat_template is not None}")

    # 采样生成：temperature/top_p 控制多样性（保持原参数）
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    # 只取「新生成」的 token，去掉 prompt 部分
    new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    print(f"  Output tokens: {len(new_tokens)}")
    # 解码为可读文本（通常含 JSON 或 markdown 代码块）
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


In [ ]:
# ========== 从模型文本抽 JSON，并做解析 / Schema 校验 ==========

# re：用正则找 ```json ... ``` 代码块
import re
# jsonschema.validate：按 JSON Schema 校验实例；ValidationError 表示不合规
from jsonschema import validate, ValidationError


# 从模型输出里提取第一个完整 JSON 对象或数组
def extract_json(text):
    # 从模型输出中提取第一个完整的 JSON 对象或数组。
    # 空输出直接放弃
    if text is None:
        return None

    # 优先匹配 markdown 代码块里的 JSON
    code_block = re.search(r"```(?:json)?\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```", text)
    if code_block:
        return code_block.group(1).strip()

    # 回退：从第一个 { 或 [ 做括号深度匹配，取最外层闭合片段
    for start, ch in enumerate(text):
        if ch not in "{[":
            continue
        close = "}" if ch == "{" else "]"
        depth = 0
        for end in range(start, len(text)):
            if text[end] == ch:
                depth += 1
            elif text[end] == close:
                depth -= 1
                if depth == 0:
                    return text[start : end + 1]
    return None


# 尝试 json.loads；成功返回 (obj, True)，失败返回 (None, False)
def validate_json(text):
    # 解析 JSON 字符串，返回 (parsed_object, is_valid) 元组。
    try:
        return json.loads(text), True
    except (json.JSONDecodeError, TypeError):
        return None, False


# 对单个对象或数组中每个元素做 schema 校验
def validate_schema(data, target_schema):
    # 检查数据是否符合目标模式。返回（符合，错误消息）。
    # 统一成列表，便于逐条校验
    items = data if isinstance(data, list) else [data]
    errors = []
    for i, item in enumerate(items):
        try:
            validate(instance=item, schema=target_schema)
        except ValidationError as e:
            # 保留英文 e.message（库返回原文）；前缀 Item i 便于定位
            errors.append(f"Item {i}: {e.message}")
    if errors:
        return False, "; ".join(errors)
    return True, None


In [ ]:
# ========== 多模型对比：生成 → 抽取 → 校验 → 释放显存 ==========

# gc：显式垃圾回收，配合 empty_cache 尽量腾出 GPU 显存给下一个模型
import gc

# 展示名 → Hub 模型 ID（常量在前面单元格已定义）
MODELS = {
    "Llama 3.1 8B": LLAMA_MODEL,
    "Qwen 2.5 Coder 7B": QWEN_MODEL,
    "Phi 4 Mini": PHI_MODEL,
}

# 收集每个模型的结果，供最后汇总表使用
results = []

# 逐个模型跑一遍完整流水线
for label, model_name in MODELS.items():
    # 分隔线 + 模型标签，方便在长日志里定位
    print(f"\n{'=' * 60}")
    print(f"  {label} ({model_name})")
    print(f"{'=' * 60}")

    try:
        # 加载量化模型与分词器
        model, tokenizer = load_model(model_name)
        # 用同一套 messages 生成原始文本
        raw_output = generate_json(model, tokenizer, messages)

        # 抽取 JSON 片段 → 解析 →（若可解析）做 schema 校验
        extracted = extract_json(raw_output)
        parsed, is_valid = validate_json(extracted)
        conforms, schema_errors = (
            validate_schema(parsed, schema) if is_valid else (False, "No valid JSON")
        )

        # 记录结构化结果（含原始输出，便于事后排查）
        results.append(
            {
                "model": label,
                "valid_json": is_valid,
                "schema_match": conforms,
                "schema_errors": schema_errors,
                "raw_output": raw_output,
                "extracted": extracted,
                "parsed": parsed,
            }
        )

        # 打印可读诊断信息
        print(f"\nRaw output:\n{raw_output}")
        print(f"\nExtracted JSON:\n{extracted}")
        print(f"\nValid JSON: {is_valid}")
        print(f"Schema match: {conforms}")
        if schema_errors:
            print(f"Schema errors: {schema_errors}")

        # 释放当前模型，避免三个 7B/8B 同时占显存
        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        # 某个模型失败时记录错误并继续下一个
        print(f"\nError: {e}")
        results.append(
            {
                "model": label,
                "valid_json": False,
                "raw_output": None,
                "extracted": None,
                "parsed": None,
                "error": str(e),
                "schema_match": False,
                "schema_errors": str(e),
            }
        )


In [ ]:
# ========== 汇总表：各模型 Valid JSON / Schema Match ==========

# 打印对比表头
print(f"\n{'=' * 60}")
print("  Model Comparison Summary")
print(f"{'=' * 60}\n")
print(f"  {'Model':<25} {'Valid JSON':<14} {'Schema Match'}")
print(f"  {'-' * 53}")
# 逐行输出每个模型的两项指标；若有 schema 错误则附带说明
for r in results:
    print(f"  {r['model']:<25} {str(r['valid_json']):<14} {r['schema_match']}")
    if r.get("schema_errors"):
        print(f"    -> {r['schema_errors']}")
# 

# ---------------------------------------------------------------------------
# kica/第6周解答.ipynb
# ---------------------------------------------------------------------------
